# Transformada de Fourier cuántica

La transformada de Fourier cuántica, QFT, es una de las construcciones más usadas en algoritmos cuánticos: aparece dentro de la estimación de fase, la factorización de Shor, y otros algoritmos que no entran en esta serie. Este notebook la construye para tres qubits y comprueba que funciona deshaciéndola con su inversa.

## Qué hace, en palabras

La QFT transforma un estado de la base computacional en una superposición donde la información original queda codificada en las fases relativas entre qubits, en vez de en los valores medidos directamente. Por eso este notebook no verifica el resultado mirando las cuentas de una única QFT: las fases no se ven así. En vez de eso, se aplica la QFT y después su inversa: si el circuito es correcto, el estado vuelve exactamente a donde empezó.

## Construir la QFT

Se construye con las puertas H y CP ya vistas en [`02a`](02a_puertas_y_construccion.ipynb), más un intercambio final de qubits con SWAP para dejarlos en el orden correcto:

In [ ]:
import math

import polypus

qc = polypus.Circuit(3)
qc.x(0)  # preparar un estado de partida concreto, |101>
qc.x(2)

# QFT hacia adelante
qc.h(0)
qc.cp(1, 0, math.pi / 2)
qc.cp(2, 0, math.pi / 4)
qc.h(1)
qc.cp(2, 1, math.pi / 2)
qc.h(2)
qc.swap(0, 2)

Cada puerta CP añade una fase que depende de la distancia entre los dos qubits: cuanto más lejos, un ángulo más pequeño. El SWAP final deshace el orden invertido que deja el resto del circuito.

Con el mismo patrón de [`02b`](02b_interoperabilidad_qiskit_qasm.ipynb), se puede ver el circuito hasta este punto:

In [ ]:
from qiskit import qasm2

qc_dibujo = qasm2.loads(
    qc.to_qasm2(), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS
)
qc_dibujo.draw("mpl")

## Deshacerla con la inversa

La QFT es una puerta unitaria, así que su inversa son las mismas puertas, en orden inverso, con los ángulos de signo contrario:

In [ ]:
qc.swap(0, 2)
qc.h(2)
qc.cp(2, 1, -math.pi / 2)
qc.h(1)
qc.cp(2, 0, -math.pi / 4)
qc.cp(1, 0, -math.pi / 2)
qc.h(0)

qc.measure_all()

## Comprobar el resultado

In [ ]:
result = polypus.run_quantum_circuit(qc, shots=2000, infrastructure="local")
print(result.counts[0])

El resultado es `'101'` en casi el 100% de las veces, el mismo estado de partida: la QFT y su inversa se cancelan exactamente, igual que la ida y vuelta de OpenQASM comprobada en [`02b`](02b_interoperabilidad_qiskit_qasm.ipynb).

## Resumen

Este notebook ha construido la transformada de Fourier cuántica para tres qubits con H, CP y SWAP, y ha comprobado que es correcta deshaciéndola con su inversa en vez de interpretar directamente sus cuentas, porque la información que codifica está en fases, no en valores medibles de forma directa.

La siguiente sección cubre los primeros algoritmos con ventaja demostrable: Deutsch-Jozsa y Grover.

## Siguiente paso

Continúa con: [`05a_deutsch_jozsa.ipynb`](05a_deutsch_jozsa.ipynb).